# MuJoCo World Visualization

This notebook keeps the same PR2 apartment scene used by the other `llmr` notebooks, but assembles a MuJoCo-safe variant of that world and visualizes it through `MujocoSim` instead of ROS2/RViz2.

## Stage 0 - Build the Same Scene for MuJoCo

The regular `llmr` notebooks use a ROS-oriented world assembly that currently breaks the MuJoCo adapter.
This cell rebuilds the same PR2 + apartment + milk + cereal scene from the same assets, but with MuJoCo-safe fixed connections.

In [1]:
from pathlib import Path

from pycram.datastructures.dataclasses import Context
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Milk, Cereal
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world_description.connections import FixedConnection
from semantic_digital_twin.world_description.world_entity import Body

REPO_ROOT = Path('/home/malineni/workingdir/cognitive_robot_abstract_machine')
PYCRAM_RESOURCES = REPO_ROOT / 'pycram' / 'resources'

PR2_URDF = 'package://iai_pr2_description/robots/pr2_with_ft2_cableguide.xacro'
APARTMENT_URDF = str(PYCRAM_RESOURCES / 'worlds' / 'apartment.urdf')
MILK_STL = str(PYCRAM_RESOURCES / 'objects' / 'milk.stl')
CEREAL_STL = str(PYCRAM_RESOURCES / 'objects' / 'breakfast_cereal.stl')


def build_mujoco_world():
    pr2_world = URDFParser.from_file(PR2_URDF).parse()
    PR2.from_world(pr2_world)

    apartment_world = URDFParser.from_file(APARTMENT_URDF).parse()
    pr2_world.merge_world(
        apartment_world,
        FixedConnection(
            parent=pr2_world.root,
            child=apartment_world.root,
        ),
    )

    apartment_root = pr2_world.get_body_by_name('apartment_root')

    milk_world = STLParser(MILK_STL).parse()
    pr2_world.merge_world(
        milk_world,
        FixedConnection(
            parent=apartment_root,
            child=milk_world.root,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                2.37,
                2.0,
                1.05,
                reference_frame=apartment_root,
            ),
        ),
    )

    cereal_world = STLParser(CEREAL_STL).parse()
    pr2_world.merge_world(
        cereal_world,
        FixedConnection(
            parent=apartment_root,
            child=cereal_world.root,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                2.37,
                1.8,
                1.05,
                reference_frame=apartment_root,
            ),
        ),
    )

    with pr2_world.modify_world():
        pr2_world.add_semantic_annotation(
            Milk(root=pr2_world.get_body_by_name('milk.stl'), _world=pr2_world)
        )
        pr2_world.add_semantic_annotation(
            Cereal(
                root=pr2_world.get_body_by_name('breakfast_cereal.stl'),
                _world=pr2_world,
            )
        )

    robot_view = pr2_world.get_semantic_annotations_by_type(PR2)[0]
    context = Context(pr2_world, robot_view)
    context.evaluate_conditions = False
    return pr2_world, robot_view, context


world, robot_view, context = build_mujoco_world()
symbol_type = Body

print('World loaded:', type(world).__name__)
print('Robot view  :', type(robot_view).__name__)
print('World root  :', world.root.name.name)
print('Bodies      :', len(list(world.bodies)))

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


World loaded: World
Robot view  : PR2
World root  : base_footprint
Bodies      : 219


## Stage 1 - Apply a Runtime MuJoCo Texture Workaround

Some imported meshes expose texture metadata without a usable filename, which currently trips the MuJoCo mesh converter.
This patch is local to the notebook process and does not modify any package source.

In [2]:
from semantic_digital_twin.adapters import multi_sim


def safe_mujoco_mesh_post_convert(self, entity, shape_props, **kwargs):
    shape_props.update(
        multi_sim.MujocoGeomConverter._post_convert(
            self,
            entity,
            shape_props,
            **kwargs,
        )
    )
    shape_props.update({'mesh': entity})

    visual = getattr(entity.mesh, 'visual', None)
    material = getattr(visual, 'material', None) if visual is not None else None
    image = getattr(material, 'image', None) if material is not None else None
    material_name = getattr(material, 'name', None) if material is not None else None
    filename = getattr(image, 'filename', None) if image is not None else None

    if (
        isinstance(visual, multi_sim.TextureVisuals)
        and isinstance(material_name, str)
        and isinstance(filename, str)
        and filename
    ):
        shape_props['texture_file_path'] = filename

    return shape_props


multi_sim.MujocoMeshConverter._post_convert = safe_mujoco_mesh_post_convert
MujocoSim = multi_sim.MujocoSim

print('Installed notebook-local MuJoCo mesh patch')

Installed notebook-local MuJoCo mesh patch


## Stage 2 - Start MuJoCo Visualization

Set `headless = False` to open the MuJoCo viewer locally.

In [3]:
import time

from physics_simulators.base_simulator import SimulatorConstraints

headless = False

multi_sim = MujocoSim(
    world=world,
    headless=headless,
    step_size=0.001,
)

constraints = SimulatorConstraints(max_number_of_steps=20000)
multi_sim.start_simulation(constraints=constraints)

print('MuJoCo simulation started')
print('Headless:', headless)
print('Scene file:', multi_sim.default_file_path)

MuJoCo simulation started
Headless: False
Scene file: /tmp/scene.xml


## Stage 3 - Monitor the Running Simulation

If the viewer opens, the PR2 apartment world should now be visible in MuJoCo.

In [4]:
time_start = time.time()

while multi_sim.is_running():
    time.sleep(0.1)
    if multi_sim.simulator.current_number_of_steps % 1000 == 0:
        print(
            'Current number of steps:',
            multi_sim.simulator.current_number_of_steps,
        )

print(f'Time elapsed: {time.time() - time_start:.2f}s')

Current number of steps: 20000
Time elapsed: 4.03s


## Stage 4 - Stop the Simulation

Run this if you stop early or want to shut MuJoCo down cleanly.

In [ ]:
if multi_sim.is_running():
    multi_sim.stop_simulation()
    print('MuJoCo simulation stopped')
else:
    print('Simulation already finished')